<a href="https://colab.research.google.com/github/cojocarucosmin/AICourseDev/blob/main/RAG_multiformat_llamaindex_Gradio_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries and download data

In [1]:
!pip install -q llama-index llama-index-llms-openai llama-index-readers-file
!pip install -q openai gradio python-docx python-pptx pandas PyMuPDF docx2txt

In [ ]:
!mkdir -p 'data/'

# Download sample data
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham_essay.txt'

In [3]:
import os
import gradio as gr
from pathlib import Path
from llama_index.llms.openai import OpenAI
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.readers.file.docs    import DocxReader, PDFReader
from llama_index.readers.file.slides  import PptxReader
from llama_index.readers.file.tabular import PandasExcelReader

In [4]:
# only for Google Colab; please comment Kaggle part in this case
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Chatbot with internal knowledge

For each chat interaction:
- first generate a standalone question from conversation context and last message, then
- query the query engine with the condensed question for a response.




In [5]:
input_dir = Path("./data/")

llm = OpenAI(model="gpt-4o", temperature=0)

# read all uploaded documents in the folder, irrespective of their format (e.g. pdf, xlsx, docx)
documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)

In [6]:
# show indexed content
shown = set()
for d in documents:
    f = d.metadata.get("file_path", d.id_)
    if f in shown:
        continue          # skip pages we've already shown
    shown.add(f)
    print(f, "\n", d.text[:100].replace("\n", " "), "\n" + "-"*20)

/content/data/Conditii contractuale Premium Care Abroad 2024.pdf 
 Condițiile generale ale contractului  de asigurare de sănătate   Premium Care Abroad  (Tratament Bol 
--------------------
/content/data/Gen_AI_Custom.docx 
 AI Generativ & Automatizare  Obiectiv: Oferirea unei perspective complete și aplicate asupra capabil 
--------------------
/content/data/Williams_Book_Store_Receipt.xlsx 
 Store Name: WILLIAMS' BOOK STORE, Established: 1908, Store Address: 708 South Pacific Avenue, San Pe 
--------------------
/content/data/paul_graham_essay.txt 
   What I Worked On  February 2021  Before college the two main things I worked on, outside of school 
--------------------


In [7]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.prompts import PromptTemplate

# Create memory buffer
memory = ChatMemoryBuffer.from_defaults(token_limit=12000)

# 3) Define your prompt (including chat history + context)
context_prompt = PromptTemplate(
    "You are a chatbot that answers the user’s questions from provided documents.\n\n"
    "Chat history:\n"
    "{chat_history_str}\n\n"
    "Relevant documents:\n"
    "{context_str}\n\n"
    "Instruction: Use the chat history above or the documents to inform your answer."
)

# 4) Build the chat engine with the temperature-tuned LLM
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    memory=memory,
    llm=llm,
    context_prompt=context_prompt,
    verbose=False,
)

### Chat with your data

In [8]:
response = chat_engine.chat("Give me details about Williams Book Store?")
response.response

"Williams' Book Store was established in 1908 and is located at 708 South Pacific Avenue, San Pedro, California, 90731. The store's phone number is 832-3631."

In [9]:
response = chat_engine.chat("Who is Paul Graham?")
response.response

'Paul Graham is an essayist, programmer, and entrepreneur known for his work in the tech industry. He co-founded Viaweb, one of the first web-based applications, which was later acquired by Yahoo. He is also a co-founder of Y Combinator, a startup accelerator that has funded numerous successful startups. Graham is recognized for his essays on various topics, including technology, startups, and programming, and has published a collection of essays titled "Hackers & Painters."'

In [10]:
response = chat_engine.chat("What do you know about Generative AI course?")
response.response

'The Generative AI course aims to provide a comprehensive and applied perspective on the capabilities of generative AI for processing text, documents, and internet searches. It focuses on conceptual understanding, practical demonstrations, and relevant case studies. Participants will learn the fundamentals of generative AI and prompt engineering, explore integration with automation processes, and work hands-on with real-world scenarios such as data search and interpretation from competing platforms, information extraction from internal documents, code/part identification from logical schemes, and using AI agents for research and data integration. The course offers both theoretical knowledge and practical skills applicable to business projects.'

### Use with Gradio

In [11]:
def predict(message, history):
    response = chat_engine.chat(message)
    return response.response

chat_ui = gr.ChatInterface(
    fn=predict,
    type="messages",
    title="AI Chatbot with Custom Knowledge",
    description="Knowledge retrieval ChatBot"
)

chat_ui.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39445001184c816d30.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Reset chat engine

In [12]:
chat_engine.reset()

## Chat Engine - ReAct Agent Mode

ReAct is an agent based chat mode built on top of a query engine over your data.

For each chat interaction, the agent enter a ReAct loop:

- first decide whether to use the query engine tool and come up with appropriate input
- (optional) use the query engine tool and observe its output
- decide whether to repeat or give final response

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# 1. Create a Query Engine from your index
query_engine = index.as_query_engine()

# 2. Wrap the Query Engine as a Tool
# This is crucial for the ReAct agent to "know" how to interact with your data.
query_engine_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="VectorStoreQueryEngine", # Give a descriptive name for your tool
        description=(
            "Useful for answering questions about the documents in the vector store. "
            "Use this tool to find specific information or answer general questions "
            "based on the provided data."
        ),
    ),
)

# 3. Initialize the ReActAgent with the tool(s) and your LLM
# The ReActAgent directly implements the ReAct loop.
# It takes a list of tools it can use.
chat_engine = ReActAgent.from_tools(
    tools=[query_engine_tool],
    llm=llm,
    verbose=True,
)

# 4. Interact with the chat engine
print("Chat engine initialized. Type your questions.")

In [15]:
response = chat_engine.chat("Use the tool to answer what Graham do in the summer of 1995?")
print(response)

> Running step 1c792128-c187-461d-b643-a6ce08ebfa98. Step input: Use the tool to answer what Graham do in the summer of 1995?
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: VectorStoreQueryEngine
Action Input: {'input': 'What did Graham do in the summer of 1995?'}
Observation: In the summer of 1995, Graham was working on Viaweb, a company he co-founded.
> Running step 05713cf5-36b6-4ef2-a3ff-59f06652de6b. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: In the summer of 1995, Graham was working on Viaweb, a company he co-founded.
In the summer of 1995, Graham was working on Viaweb, a company he co-founded.


In [16]:
response = chat_engine.chat("Is there any mention about 1995?")
print(response)

> Running step 175530b8-3d92-4066-a09b-a2b9e214d10d. Step input: Is there any mention about 1995?
Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: VectorStoreQueryEngine
Action Input: {'input': '1995'}
Observation: 1995 was a significant year in the context provided as it marked the beginning of the journey described by the author. It was the year when the author and his team started their venture, Viaweb, which eventually led to a successful acquisition by Yahoo.
> Running step 0deef5c4-2bea-4e25-87f5-7df522ecc246. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: Yes, 1995 was a significant year as it marked the beginning of the author's journey with the venture Viaweb, which eventually led to a successful acquisition by Yahoo.
Yes, 1995 was a significant year as it marked the beginning of the author's journey with the venture Viaweb, which eventually l

In [17]:
response = chat_engine.chat("Who is Klaus Iohannis?")
print(response)

> Running step 59c9d604-215a-437d-a46d-b049a8c7f5f3. Step input: Who is Klaus Iohannis?
Thought: (Implicit) I can answer without any more tools!
Answer: Klaus Iohannis is a Romanian politician who has been serving as the President of Romania since December 2014. Before becoming president, he was the mayor of Sibiu, a city in Transylvania, Romania. Iohannis is a member of the National Liberal Party (PNL) and is known for his pro-European Union stance and efforts to combat corruption in Romania.
Klaus Iohannis is a Romanian politician who has been serving as the President of Romania since December 2014. Before becoming president, he was the mayor of Sibiu, a city in Transylvania, Romania. Iohannis is a member of the National Liberal Party (PNL) and is known for his pro-European Union stance and efforts to combat corruption in Romania.
